# Instructions

Make a copy of this colab. Answer all questions by writing code. Run all cells and save output. Download .ipynb and submit it in canvas.

Please check back often for any updates.

2025/10/25 10pm: Initial version.

2025/10/26 3:15pm: Added a remark on keeping number of trainable params small.

2025/10/27 11:16pm: Added a question about defining your own custom MySparseTopkCatAccuracy and added points breakdown.


# Imports

In [ ]:
from packaging import version

import sklearn
print('sklearn.__version__:', sklearn.__version__)
assert version.parse(sklearn.__version__) >= version.parse('1.0.1')

import numpy as np
np.random.seed(1237)  # 42 or some prime number

import pandas as pd
import matplotlib.pyplot as plt
import functools

import tensorflow as tf

print('tf.__version__:', tf.__version__)
print('GPUs:', len(tf.config.list_physical_devices('GPU')))

keras = tf.keras
layers = keras.layers


sklearn.__version__: 1.6.1
tf.__version__: 2.19.0
GPUs: 1


# Dataset

Download cifar-10 dataset from keras.

In [ ]:
from tensorflow.keras.datasets import cifar10
(X_train, y_train), (X_test, y_test) = cifar10.load_data()


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [ ]:
X_train, X_valid = X_train[:-5000], X_train[-5000:]
y_train, y_valid = y_train[:-5000], y_train[-5000:]

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)
print(X_valid.shape)
print(y_valid.shape)

(45000, 32, 32, 3)
(45000, 1)
(10000, 32, 32, 3)
(10000, 1)
(5000, 32, 32, 3)
(5000, 1)


# Part 1 (6 points)

* (1 point) Define your own ResidualUnit block using subclassing api.
* (1 point) Use 4 such blocks to build a CNN model and train it on cifar10.
* (1 point) Be sure to use data augmentation for your training data.
* You would also need to use one Conv2D layer on the input side and one GlobalAveragePooling2D/Dense at the top to predict 10 classes.
* (1 point) How many trainable params are there in your model? Adjust your network so that number of trainable params is ~100k-120k.
* (1 point) Also measure accuracy on test set using built-in keras accuracy function.
* (1 point) Define your own custom streaming metric MySparseTopkCatAccuracy. Use it to compute k=3 accuracy for your model on the test set. You can use built-in keras.metrics.SparseTopKCategoricalAccuracy(k=3) for 0 points. You should get >80% accuracy for k=3.


In [ ]:
#Question1
from functools import partial
DefaultConv2D = partial(keras.layers.Conv2D, kernel_size=3, strides=1,
                        padding="SAME", use_bias=False)

class ResidualUnit(keras.layers.Layer):
    def __init__(self, filters, strides=1, activation="relu", **kwargs):
        super().__init__(**kwargs)
        self.activation = keras.activations.get(activation)
        self.main_layers = [
            DefaultConv2D(filters, strides=strides),
            keras.layers.BatchNormalization(),
            self.activation,
            DefaultConv2D(filters),
            keras.layers.BatchNormalization()]
        self.skip_layers = []
        if strides > 1:
            self.skip_layers = [
                DefaultConv2D(filters, kernel_size=1, strides=strides),
                keras.layers.BatchNormalization()]

    def call(self, inputs):
        Z = inputs
        for layer in self.main_layers:
            Z = layer(Z)
        skip_Z = inputs
        for layer in self.skip_layers:
            skip_Z = layer(skip_Z)
        return self.activation(Z + skip_Z)

In [ ]:
#Question3
data_augmentation = keras.Sequential([
	layers.RandomFlip(mode='horizontal'),
	layers.RandomTranslation(height_factor=0.05, width_factor=0.05, fill_mode='constant', fill_value=0),
	layers.RandomRotation(factor=0.05, fill_mode='constant', fill_value=0),
	layers.RandomContrast(factor=0.1),
], name='aug')

In [ ]:
import tensorflow as tf
#Question7
class MySparseTopKCategoricalAccuracy(tf.keras.metrics.Metric):
    def __init__(self, k=3, name="my_sparse_top_k_categorical_accuracy", **kwargs):
        super().__init__(name=name, **kwargs)
        self.k = k
        self.t = self.add_weight(name="t", initializer="zeros")
        self.c = self.add_weight(name="c", initializer="zeros")

    def update_state(self, y, y_p, sample_weight=None):
        y = tf.cast(tf.reshape(y, [-1]), tf.int32)
        k_top = tf.nn.in_top_k(predictions=y_p, targets=y,k=self.k)
        #print("Top K",k_top)
        k_top = tf.cast(k_top, tf.float32)

        if sample_weight is not None:
            sample_weight = tf.cast(sample_weight, tf.float32)
            k_top *= sample_weight

        self.c.assign_add(tf.reduce_sum(k_top))
        self.t.assign_add(tf.cast(tf.size(y), tf.float32))

    def result(self):
        return self.c / self.t

    def reset_states(self):
        self.c.assign(0.0)
        self.t.assign(0.0)


In [ ]:
#Question2
model1 = keras.models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,
    DefaultConv2D(64), #Question4
    #BatchNomalization(),
    #layers.Activation("relu"), #not mandatory

    ResidualUnit(64, strides=2),
    ResidualUnit(24, strides=2),
    ResidualUnit(16, strides=2),
    ResidualUnit(16, strides=2),
    keras.layers.GlobalAvgPool2D(), #Question4
])
model1.add(keras.layers.Dense(10, activation="softmax"))
model1.summary() #Question5
c

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ aug (Sequential)                │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_26 (Conv2D)              │ (None, 32, 32, 64)     │         1,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_unit_8 (ResidualUnit)  │ (None, 16, 16, 64)     │        78,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_unit_9 (ResidualUnit)  │ (None, 8, 8, 24)       │        20,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_unit_10 (ResidualUnit) │ (None, 4, 4, 16)       │         6,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_unit_11 (ResidualUnit) │ (None, 2, 2, 16)       │         5,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 16)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           170 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 112,714 (440.29 KB)

 Trainable params: 111,994 (437.48 KB)

 Non-trainable params: 720 (2.81 KB)

In [ ]:
metric1 = MySparseTopKCategoricalAccuracy(k=3)

model1.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=[metric1])
history = model1.fit(X_train, y_train, epochs=6, validation_data=(X_valid, y_valid))

Epoch 1/6
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 46s 24ms/step - loss: 1.8372 - my_sparse_top_k_categorical_accuracy: 0.6455 - val_loss: 1.5038 - val_my_sparse_top_k_categorical_accuracy: 0.7886
Epoch 2/6
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 33s 23ms/step - loss: 1.3942 - my_sparse_top_k_categorical_accuracy: 0.8156 - val_loss: 1.2590 - val_my_sparse_top_k_categorical_accuracy: 0.8512
Epoch 3/6
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 33s 23ms/step - loss: 1.2541 - my_sparse_top_k_categorical_accuracy: 0.8479 - val_loss: 1.2651 - val_my_sparse_top_k_categorical_accuracy: 0.8604
Epoch 4/6
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 32s 23ms/step - loss: 1.1524 - my_sparse_top_k_categorical_accuracy: 0.8725 - val_loss: 1.2370 - val_my_sparse_top_k_categorical_accuracy: 0.8652
Epoch 5/6
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 33s 23ms/step - loss: 1.0842 - my_sparse_top_k_categorical_accuracy: 0.8830 - val_loss: 1.1298 - val_my_sparse_top_k_categorical_accuracy: 0.8808
Epoch 6/6
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 33s 23ms/step - loss: 1.

In [ ]:
#Question7
acc1 = MySparseTopKCategoricalAccuracy(k=3)

y_p = model1.predict(X_test)

acc1.update_state(y_test, y_p)

print("MySparseTopKCategoricalAccuracy K=3:", acc1.result().numpy())

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
MySparseTopKCategoricalAccuracy K=3: 0.8775


In [ ]:
#Question6
model2.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])

history_2 = model2.fit(X_train, y_train, epochs=20, validation_data=(X_valid, y_valid))


Epoch 1/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 42s 24ms/step - accuracy: 0.7069 - loss: 0.8387 - val_accuracy: 0.6996 - val_loss: 0.8667
Epoch 2/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 41s 29ms/step - accuracy: 0.7184 - loss: 0.8038 - val_accuracy: 0.6722 - val_loss: 0.9690
Epoch 3/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 34s 24ms/step - accuracy: 0.7230 - loss: 0.7895 - val_accuracy: 0.7146 - val_loss: 0.8181
Epoch 4/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 34s 24ms/step - accuracy: 0.7297 - loss: 0.7661 - val_accuracy: 0.7388 - val_loss: 0.7496
Epoch 5/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 44s 31ms/step - accuracy: 0.7361 - loss: 0.7493 - val_accuracy: 0.7194 - val_loss: 0.8516
Epoch 6/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 34s 24ms/step - accuracy: 0.7407 - loss: 0.7365 - val_accuracy: 0.7164 - val_loss: 0.8332
Epoch 7/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 42s 25ms/step - accuracy: 0.7425 - loss: 0.7340 - val_accuracy: 0.7242 - val_loss: 0.8005
Epoch 8/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 36s 25ms/step - accuracy: 0.7452 -

In [ ]:
#Question6
test_loss_2, test_acc_2 = model2.evaluate(X_test, y_test)
print("Test loss:", test_loss_2)
print("Test accuracy:", test_acc_2)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7555 - loss: 0.7214
Test loss: 0.7232251763343811
Test accuracy: 0.7519999742507935


# Part 2 (4 points)

* (0.5 point) Get pretrained ResNet50 with include_top=False, freeze the layers.
* (1 point) add 1 new residual block on top and train it.
* Be sure to use data augmentation for your training data.
* (0.5 point) You would need to add keras.layers.UpSampling2D(size=(7,7)) on the input side to convert cifar10's (32,32) to resnet's (224,224).
* (0.5 point) You would also need resnet50.preprocess_input on input.
* You would still need GlobalAveragePooling2D/Dense at top to predict 10 classes.
* (1 point) How many trainable params are there in your model? Adjust your network so that number of trainable params is ~100k-120k.
* (0.5 point) Measure accuracy and MySparseTopkCatAccuracy(k=3). You should get >95% for k=3.


In [ ]:
#Question2

from functools import partial
from tensorflow import keras
from tensorflow.keras import layers

DefaultConv2D_1 = partial(keras.layers.Conv2D, kernel_size=1, strides=2,
                          padding="same", use_bias=False)

DefaultConv2D_2 = partial(keras.layers.Conv2D, kernel_size=1, strides=2,
                          padding="same", use_bias=False)

class ResidualUnit_1(keras.layers.Layer):
    def __init__(self, bottleneck_filters=20, activation="relu", **kwargs):
        super().__init__(**kwargs)
        self.activation_fn = keras.activations.get(activation)

        self.main_layers = [
            DefaultConv2D_2(bottleneck_filters),
            layers.BatchNormalization(),
            self.activation_fn,
            DefaultConv2D_1(bottleneck_filters),
            layers.BatchNormalization(),
            self.activation_fn,
            DefaultConv2D_2(2048),
            layers.BatchNormalization()
        ]

    def call(self, inputs):
        Z = inputs
        for layer in self.main_layers:
            Z = layer(Z)
        return self.activation_fn(Z + inputs)


In [ ]:
#Question1

resnet = keras.applications.resnet50.ResNet50(include_top=False, weights='imagenet', input_shape=(224, 224, 3), name='resnet')
resnet.trainable = False
preprocess_fn = lambda x: keras.applications.resnet50.preprocess_input(x)
model4 = keras.models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation, #Question3
    layers.UpSampling2D(size=(7, 7), interpolation='bilinear'), #Question4

    layers.Lambda(lambda y: keras.applications.resnet50.preprocess_input(y)), #Question5

    resnet,
    ResidualUnit_1(bottleneck_filters=20), #Question2
    layers.GlobalAvgPool2D(),  #Question6
    layers.Dense(10, activation="softmax")
])

model4.summary() #Question7

Model: "sequential_38"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ aug (Sequential)                │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_34 (UpSampling2D) │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_34 (Lambda)              │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet (Functional)             │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_unit_1_35              │ (None, 7, 7, 2048)     │        90,672 │
│ (ResidualUnit_1)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_38     │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_38 (Dense)                │ (None, 10)             │        20,490 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,698,874 (90.40 MB)

 Trainable params: 106,986 (417.91 KB)

 Non-trainable params: 23,591,888 (90.00 MB)

In [ ]:
#Question8

metric4 = MySparseTopKCategoricalAccuracy(k=3)

model4.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=[metric4,"accuracy"])
history4 = model4.fit(X_train, y_train, epochs=5, validation_data=(X_valid, y_valid))

Epoch 1/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 215s 147ms/step - accuracy: 0.6425 - loss: 1.0526 - my_sparse_top_k_categorical_accuracy: 0.8715 - val_accuracy: 0.8590 - val_loss: 0.5273 - val_my_sparse_top_k_categorical_accuracy: 0.9760
Epoch 2/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 205s 145ms/step - accuracy: 0.8060 - loss: 0.5531 - my_sparse_top_k_categorical_accuracy: 0.9648 - val_accuracy: 0.8748 - val_loss: 0.4521 - val_my_sparse_top_k_categorical_accuracy: 0.9774
Epoch 3/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 205s 146ms/step - accuracy: 0.8244 - loss: 0.5092 - my_sparse_top_k_categorical_accuracy: 0.9724 - val_accuracy: 0.8788 - val_loss: 0.5143 - val_my_sparse_top_k_categorical_accuracy: 0.9828
Epoch 4/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 205s 145ms/step - accuracy: 0.8361 - loss: 0.4759 - my_sparse_top_k_categorical_accuracy: 0.9724 - val_accuracy: 0.8918 - val_loss: 0.3928 - val_my_sparse_top_k_categorical_accuracy: 0.9852
Epoch 5/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 205s 146ms/step - accuracy: 0.8379 

In [ ]:
acc4 = MySparseTopKCategoricalAccuracy(k=3)

y_p4 = model4.predict(X_test)

acc4.update_state(y_test, y_p4)

print("MySparseTopKCategoricalAccuracy K=3:", acc4.result().numpy())

313/313 ━━━━━━━━━━━━━━━━━━━━ 42s 130ms/step
MySparseTopKCategoricalAccuracy K=3: 0.9857


In [ ]:
#Question6
acc = model4.evaluate(X_test, y_test)

print("Test accuracy:", acc[2])

313/313 ━━━━━━━━━━━━━━━━━━━━ 37s 119ms/step - accuracy: 0.8875 - loss: 0.3863 - my_sparse_top_k_categorical_accuracy: 0.9853
Test accuracy: 0.8860999941825867
